# Traceprop review runs (items 1, 2, 8)

Produces the result files `docs/mlsys/REVIEW_SCAFFOLD.md` needs. Run on an **A100** runtime;
job 4 is CPU-only and runs fine in the same A100 session. **Item 9 (rank agreement) is excluded**:
it needs a score-matrix step not yet wired.

Each run writes to `experiments/results/`; the last cell copies them to Drive. Download these back
into your local `experiments/results/`, then tell the assistant:

- `exp36_memory_storage_pythia1b.json`
- `exp35_inlineprecond_overhead_hf_pythia7b_split.json`
- `exp35_overhead_1b_k8.json`, `exp35_overhead_1b_k32.json`, `exp35_overhead_1b_k64.json`
- `exp35_ksweep_seed0.npz` ... `exp35_ksweep_seed4.npz`

`exp36` is a new script not yet run on GPU; if it errors, send the traceback.

In [3]:
!python --version
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU-only'
print('GPU:', name)
if 'A100' not in name:
    print('WARNING: jobs 1-3 expect an A100 (40GB); results on a different GPU are not comparable '
          'to the paper. Job 4 (quality sweep) is CPU-only and is fine on any runtime.')

# Install matches the working stage2 notebook: use Colab's current numpy/transformers/peft,
# drop torchao (its version check breaks peft LoRA dispatch), install logix-ai for the LogIX arm.
!pip -q install datasets scipy scikit-learn accelerate
!pip -q uninstall -y torchao 2>/dev/null
!pip -q install --ignore-requires-python logix-ai
import numpy, transformers, peft
print('numpy', numpy.__version__, '| transformers', transformers.__version__, '| peft', peft.__version__)


Python 3.13.15
NVIDIA A100-SXM4-80GB, 81920 MiB
GPU: NVIDIA A100-SXM4-80GB
numpy 2.1.3 | transformers 5.20.1 | peft 0.21.2


In [4]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/traceprop_runs', exist_ok=True)


Mounted at /content/drive


In [5]:
# Repo is public; press Enter at the prompt for an anonymous clone, or paste a token.
import getpass
TOKEN = getpass.getpass('GitHub token (Enter for public clone): ').strip()
url = f'https://{TOKEN}@github.com/AmitoVrito/Traceprop.git' if TOKEN else 'https://github.com/AmitoVrito/Traceprop.git'
!git clone -q {url} /content/Traceprop || (cd /content/Traceprop && git pull -q)
%cd /content/Traceprop
!git log --oneline -1
!pip -q install -e .
%cd /content/Traceprop/experiments


GitHub token (Enter for public clone): ··········
/content/Traceprop
fddd8f4 (HEAD -> main, origin/main, origin/HEAD) Add review_runs_colab.ipynb: runs items 1/2/8 (exp36, 6.9B split, k-sweep)
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for traceprop (pyproject.toml) ... done
/content/Traceprop/experiments


In [6]:
!mkdir -p results
!cp /content/drive/MyDrive/traceprop_runs/*.json results/ 2>/dev/null
!cp /content/drive/MyDrive/traceprop_runs/*.npz results/ 2>/dev/null
!ls results/


exp25b_overhead_vs_modelsize.json
exp25_hf_EleutherAI_pythia-1b_track1.json
exp25_hf_EleutherAI_pythia-2.8b_track1.json
exp25_hf_EleutherAI_pythia-410m_track1.json
exp25_hf_gpt2_track1.json
exp25_hf_pythia-1b_track1.json
exp25_hf_pythia-2.8b_track1.json
exp25_hf_pythia-410m_track1.json
exp25_pythia1b_track0_factored_kfac8.json
exp25_pythia1b_track1_factored_kfac8.json
exp25_pythia1b_track1_matched256.json
exp25_pythia1b_track6_factored_kfac8.json
exp25_pythia1b_track6_matched1536.json
exp25_tiny_tiny-gpt.json
exp25_tiny_tiny-gpt_track1.json
exp26_headtohead_repeats20_summary.json
exp26_hf_gpt2_track1.json
exp26_tiny_tiny-gpt_track1.json
exp27_hf_gpt2.json
exp27_tiny_tiny-clf.json
exp28_hf_gpt2.json
exp28_tiny_frozen-synth.json
exp29_hf_gpt2.json
exp29_hf_gpt2_rerun.json
exp29_hf_gpt2_rerun_raw.npz
exp29_tiny_tiny-clf.json
exp30_same_setting_overhead.json
exp31_logix_comparison.json
exp31_logix_hf_gpt2.json
exp31_logix_tiny_tiny-clf.json
exp31_pythia1b_track0_fixed2.json
exp31_pythia1b_

## Job 1 — memory/storage (item 1). A100, ~5 min.

In [7]:
!python exp36_memory_storage.py --model EleutherAI/pythia-1b --device cuda \
    --steps 200 --batch 8 --seq 128 --kfac 8 --track 0 \
    --out results/exp36_memory_storage_pythia1b.json


config.json: 100% 569/569 [00:00<00:00, 2.95MB/s]
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!

model.safetensors: downloading bytes:   0% 0.00/2.09G [00:00<?, ?B/s]
model.safetensors: downloading bytes:  10% 217M/2.09G [00:01<00:07, 263MB/s, 13.4MB/s  ]
model.safetensors: downloading bytes:  14% 285M/2.09G [00:02<00:06, 288MB/s, 24.7MB/s  ]
model.safetensors: downloading bytes:  29% 608M/2.09G [00:02<00:02, 568MB/s, 48.0MB/s  ]
model.safetensors: reconstructing file:  22% 469M/2.09G [00:02<00:06, 266MB/s, 25.3MB/s  ]
model.safetensors: downloading bytes:  39% 823M/2.09G [00:02<00:02, 551MB/s, 67.2MB/s  ]
model.safetensors: downloading bytes:  49% 1.02G/2.09G [00:03<00:01, 594MB/s, 83.3MB/s  ]
model.safetensors: downloading bytes:  53% 1.10G/2.09G [00:03<00:01, 531MB/s, 91.0MB/s  ]
model.safetensors: downloading bytes:  62% 1.29G/2.09G [00:03<00:01, 504MB/s,  105MB/s  ]
model.safetensors: downloading bytes:  76% 1.59G/2.09G [00:04<00:00, 569MB/s,  124MB/s  ]
model.s

## Job 2 — 6.9B logging/covariance split (item 2). A100, ~15-20 min.

Same as the saved 6.9B run but WITHOUT `--skip_log_only`, so the logging-only arm is timed.

In [8]:
!python exp35_inlineprecond_overhead_hf.py --model EleutherAI/pythia-6.9b \
    --device cuda --dtype auto --track 0 --kfac 8 --steps 50 --batch 8 --seq 128 \
    --repeats 5 --with_logix --skip_logix_pca \
    --out results/exp35_inlineprecond_overhead_hf_pythia7b_split.json


config.json: 100% 571/571 [00:00<00:00, 3.18MB/s]
model.safetensors.index.json: 100% 44.0k/44.0k [00:00<00:00, 108MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0% 0/2 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/3.94G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   3% 379M/13.8G [00:02<01:04, 209MB/s,  185MB/s  ]
Reconstructing (incomplete total...):  11% 1.51G/13.8G [00:05<00:31, 391MB/s,  314MB/s  ]
Reconstructing (incomplete total...):  15% 2.12G/13.8G [00:07<00:34, 342MB/s,  285MB/s  ]
Reconstructing (incomplete total...):  17% 2.29G/13.8G [00:08<00:48, 240MB/s,  399MB/s  ]
Reconstructing (incomplete total...):  27% 3.76G/13.8G [00:10<00:16, 608MB/s,  462MB/s  ]
Reconstructing (incomplete total...):  29% 4.04G/13.8G [00:11<00:17, 570MB/s,  643MB/s  ]
Reconstructing (incomplete total...):  31% 4.30G/13.8G [00:11<00:16, 584MB/s,  672MB/s  ]
Reconstructing (incomplete total...):  35% 4.82G

## Job 3 — k-sweep overhead at 1B (item 8, overhead half). A100, ~30-40 min.

In [9]:
for k in [8, 32, 64]:
    print(f'=== k={k} ===')
    !python exp35_inlineprecond_overhead_hf.py --model EleutherAI/pythia-1b \
        --device cuda --dtype fp32 --track 0 --kfac {k} --steps 200 --batch 8 \
        --seq 128 --repeats 10 --with_logix \
        --out results/exp35_overhead_1b_k{k}.json


=== k=8 ===
Loading weights: 100% 195/195 [00:00<00:00, 5267.33it/s]
[transformers] GPTNeoXForSequenceClassification LOAD REPORT from: EleutherAI/pythia-1b
Key              | Status     | 
-----------------+------------+-
embed_out.weight | UNEXPECTED | 
score.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
tokenizer_config.json: 100% 396/396 [00:00<00:00, 2.02MB/s]
tokenizer.json: 100% 2.11M/2.11M [00:00<00:00, 142MB/s]
special_tokens_map.json: 100% 99.0/99.0 [00:00<00:00, 696kB/s]
[overhead-hf] EleutherAI/pythia-1b on cuda, dtype=fp32 (requested=fp32), 34 tracked layers, kfac=8, 200 steps x b8 x s128, 10 repeats, grouped_dispatch=False
<sys>:0: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs r

## Job 4 — k-sweep quality (item 8, quality half). CPU-only; runs in the same session, ~15-25 min.

In [10]:
for s in range(5):
    print(f'=== seed {s} ===')
    !python exp35_logix_lds.py --backend tiny --data synthetic --device cpu \
        --track 0 --factored --inline_precond --tune_logix --lora_init random \
        --kfac_values 8,32,64 --seed {s} \
        --out results/exp35_ksweep_seed{s}.npz


=== seed 0 ===
[exp35] training target model (400 examples) ...
[exp35] target test accuracy: 0.7900
[2026-10-19 20:22:03] [INFO] blocks.0.q.lora_A: Linear(in_features=128, out_features=8, bias=False)
[2026-10-19 20:22:03] [INFO] blocks.0.q.lora_B: Linear(in_features=8, out_features=128, bias=False)
[2026-10-19 20:22:03] [INFO] blocks.0.k.lora_A: Linear(in_features=128, out_features=8, bias=False)
[2026-10-19 20:22:03] [INFO] blocks.0.k.lora_B: Linear(in_features=8, out_features=128, bias=False)
[2026-10-19 20:22:03] [INFO] blocks.0.v.lora_A: Linear(in_features=128, out_features=8, bias=False)
[2026-10-19 20:22:03] [INFO] blocks.0.v.lora_B: Linear(in_features=8, out_features=128, bias=False)
[2026-10-19 20:22:03] [INFO] blocks.0.o.lora_A: Linear(in_features=128, out_features=8, bias=False)
[2026-10-19 20:22:03] [INFO] blocks.0.o.lora_B: Linear(in_features=8, out_features=128, bias=False)
[2026-10-19 20:22:03] [INFO] blocks.0.fc1.lora_A: Linear(in_features=128, out_features=8, bias=Fals

## Save results to Drive

Then download these from `MyDrive/traceprop_runs/` into your local `experiments/results/`.

In [11]:
!cp results/exp36_memory_storage_pythia1b.json /content/drive/MyDrive/traceprop_runs/ 2>/dev/null
!cp results/exp35_inlineprecond_overhead_hf_pythia7b_split.json /content/drive/MyDrive/traceprop_runs/ 2>/dev/null
!cp results/exp35_overhead_1b_k*.json /content/drive/MyDrive/traceprop_runs/ 2>/dev/null
!cp results/exp35_ksweep_seed*.npz /content/drive/MyDrive/traceprop_runs/ 2>/dev/null
print('copied to Drive:')
!ls -la /content/drive/MyDrive/traceprop_runs/ | grep -E 'exp36|pythia7b_split|overhead_1b_k|ksweep'


copied to Drive:
-rw-------  1 root root    1497 Oct 19 20:22 exp35_inlineprecond_overhead_hf_pythia7b_split.json
-rw-------  1 root root    1967 Oct 19 20:22 exp35_overhead_1b_k32.json
-rw-------  1 root root    1968 Oct 19 20:22 exp35_overhead_1b_k64.json
-rw-------  1 root root    1958 Oct 19 20:22 exp35_overhead_1b_k8.json
-rw-------  1 root root    1212 Oct 19 20:22 exp36_memory_storage_pythia1b.json
